In [1]:
import numpy as np
import random

In [ ]:
def fitness_function(solution):
    """
    Calculates the fitness of a tube design. A lower score is better.
    
    Args:
        solution (list): A list containing [radius, height].
        
    Returns:
        float: The fitness score (area + penalty).
    """
    r, h = solution
    
    # A solution is invalid if dimensions are not positive
    if r <= 0 or h <= 0:
        return float('inf') # Return a very large number (infinity)
        
    # --- 1. Objective: Calculate the surface area ---
    surface_area = np.pi * r**2 + 2 * np.pi * r * h
    
    # --- 2. Constraint: Check the volume ---
    volume = np.pi * r**2 * h
    
    # --- 3. Penalty: Apply if constraint is violated ---
    penalty = 0
    required_volume = 200
    
    if volume < required_volume:
        # Apply a heavy penalty proportional to the shortfall
        penalty_factor = 0
        penalty = penalty_factor * (required_volume - volume)
        
    # The final fitness is the area plus any penalty
    return surface_area + penalty

# --- Example Usage ---

# An optimal solution (found by an optimization algorithm)
# Here, radius equals height
optimal_solution = [3.9929, 3.9929] 
# Volume = pi * 3.9929^2 * 3.9929 = 200.0

# A poor solution that violates the volume constraint
poor_solution = [3, 3]
# Volume = pi * 3^2 * 3 = 84.8

print(f"Fitness of the optimal solution: {fitness_function(optimal_solution):.2f}")
print(f"Fitness of the poor solution: {fitness_function(poor_solution):.2f}")

Fitness of the optimal solution: 218.52
Fitness of the poor solution: 1151854.81


In [3]:
# --- 1. Define the Problem ---
VOLUME_CONSTRAINT = 200.0

def calculate_area(r, h):
    """Calculates the surface area of the open-top cylinder."""
    return np.pi * r**2 + 2 * np.pi * r * h

def calculate_volume(r, h):
    """Calculates the volume of the cylinder."""
    return np.pi * r**2 * h

# --- 2. Define the Fitness Function ---
def fitness_function(individual):
    """
    Calculates the fitness of an individual [radius, height].
    Returns 0 for invalid designs, otherwise returns 1/Area.
    """
    radius, height = individual
    if radius <= 0 or height <= 0:
        return 0
        
    volume = calculate_volume(radius, height)
    
    # Penalize solutions that do not meet the volume constraint
    if volume < VOLUME_CONSTRAINT:
        return 0
    else:
        # For valid solutions, fitness is the inverse of the area
        area = calculate_area(radius, height)
        return 1.0 / area

# --- 3. Genetic Algorithm Components ---

def tournament_selection(population, scores, k=3):
    """Selects a parent using k-tournament selection."""
    # Get k random contestants
    selection_ix = np.random.randint(len(population))
    for _ in range(k - 1):
        ix = np.random.randint(len(population))
        # Check if this contestant is better than the current best
        if scores[ix] > scores[selection_ix]:
            selection_ix = ix
    return population[selection_ix]

def crossover(p1, p2, r_cross=0.9):
    """Performs crossover between two parents to create two children."""
    c1, c2 = p1.copy(), p2.copy()
    if random.random() < r_cross:
        # Arithmetic crossover for radius and height
        c1 = [p1[0] * 0.7 + p2[0] * 0.3, p1[1] * 0.7 + p2[1] * 0.3]
        c2 = [p1[0] * 0.3 + p2[0] * 0.7, p1[1] * 0.3 + p2[1] * 0.7]
    return [c1, c2]

def mutation(individual, r_mut=0.1):
    """Performs mutation on an individual."""
    if random.random() < r_mut:
        # Add a small random value to one of the genes
        gene_to_mutate = random.randint(0, 1)
        mutation_value = np.random.normal(0, 0.5) # Small change
        individual[gene_to_mutate] += mutation_value
        # Ensure dimensions don't become negative
        if individual[gene_to_mutate] < 0:
            individual[gene_to_mutate] = 0.1

# --- 4. Main Algorithm ---

# Hyperparameters
POPULATION_SIZE = 100
NUM_GENERATIONS = 50
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.2
DIM_RANGE = (0.1, 20.0) # Search range for radius and height

# Initialization
population = [[random.uniform(*DIM_RANGE), random.uniform(*DIM_RANGE)] for _ in range(POPULATION_SIZE)]

best_individual = None
best_fitness = -1

print("Running Genetic Algorithm...")

for gen in range(NUM_GENERATIONS):
    # Calculate fitness for the current population
    fitness_scores = [fitness_function(ind) for ind in population]

    # Find the best individual in this generation
    for i in range(POPULATION_SIZE):
        if fitness_scores[i] > best_fitness:
            best_fitness = fitness_scores[i]
            best_individual = population[i]
            
    # Create the next generation
    children = []
    while len(children) < POPULATION_SIZE:
        # Selection
        parent1 = tournament_selection(population, fitness_scores)
        parent2 = tournament_selection(population, fitness_scores)
        
        # Crossover
        for child in crossover(parent1, parent2, CROSSOVER_RATE):
            # Mutation
            mutation(child, MUTATION_RATE)
            children.append(child)
            
    population = children

# --- 5. Display Results ---
best_radius, best_height = best_individual
final_area = calculate_area(best_radius, best_height)
final_volume = calculate_volume(best_radius, best_height)

print("\n--- Optimal Design Found ---")
print(f"Radius (r): {best_radius:.4f}")
print(f"Height (h): {best_height:.4f}")
print("----------------------------")
print(f"Material Used (Area): {final_area:.4f}")
print(f"Liquid Capacity (Volume): {final_volume:.4f}")

# Analytical check: For an open-top cylinder, minimal area occurs when radius = height.
# r = (200/pi)^(1/3) ~= 3.99
# This confirms the GA result is accurate.

Running Genetic Algorithm...

--- Optimal Design Found ---
Radius (r): 3.9317
Height (h): 4.1195
----------------------------
Material Used (Area): 150.3294
Liquid Capacity (Volume): 200.0563
